In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS

# 10 - Merge RegBL and Hydrogeo (hydrogeo_main_delimit)

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `08_OFEV_hydrogeo_main_delimit`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Hydrogeo

In [ ]:
gdf_hydro = gpd.read_file(f"{ROOT}data/raw/08_OFEV_hydrogeo_main_delimit/HydrogeologischeSkizze.shp")

gdf_hydro.head()

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.6.csv', index_col=0)

df_regbl.head()

## Keep only needed columns and rename them

In [ ]:
COLS_TO_KEEP = [
 'Name',
 'aquifertyp',
 'Shape_Leng',
 'Shape_Area',
 'flaeche',
 'geometry'
]

gdf_hydro = gdf_hydro[COLS_TO_KEEP]

gdf_hydro = gdf_hydro.rename(columns={
    'Name': 'Hydro - Nom',
    'aquifertyp': 'Hydro - Type Aquifère',
    'Shape_Leng': 'Hydro - Longueur',
    'Shape_Area': 'Hydro - Surface',
    'flaeche': 'Hydro - Zone'
})

## Convert current dataset into GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Perform spatial joint

In [ ]:
df_regbl = gpd.sjoin(df_regbl, gdf_hydro, how="left", predicate="within")

## Convert current dataset back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.shape

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.7.csv")